<a href="https://colab.research.google.com/github/chughtaianosh-boop/evolvix-aiml-internship-anoshchughtai/blob/main/week-4/task-1/week_4_task_1_Anosh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install required packages
!pip install fastapi uvicorn nest_asyncio pydantic transformers torch
!npm install -g localtunnel

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from transformers import pipeline
import threading
import time
import subprocess

# 2. Apply nest_asyncio to allow running Uvicorn inside the Jupyter Notebook environment
nest_asyncio.apply()

# 3. Initialize FastAPI App
app = FastAPI(
    title="IMDb Movie Review Sentiment Analysis API",
    description="FastAPI wrapper for fine-tuned Longformer sentiment classification model.",
    version="1.0"
)

# 4. Load the Hugging Face Model Pipeline
print("🤖 Loading model pipeline, please wait...")
try:
    pipe = pipeline(
        "text-classification",
        model="ahmed792002/Finetuning_Longformer_IMDb_movie_reviews_Classification"
    )
    print("✅ Model successfully loaded!")
except Exception as e:
    print(f"❌ Error loading model: {str(e)}")
    pipe = None

# 5. Define Pydantic Input Schema for Validation
class ReviewRequest(BaseModel):
    review: str = Field(..., min_length=1, description="The movie review text cannot be empty.")

# 6. Define prediction endpoint with robust error handling
@app.post("/predict")
def predict_sentiment(request: ReviewRequest):
    if pipe is None:
        raise HTTPException(status_code=500, detail="Model pipeline is not initialized properly.")

    try:
        # Clean and validate text
        text = request.review.strip()
        if not text:
            raise HTTPException(status_code=400, detail="Review text must contain non-whitespace characters.")

        # Get model prediction
        result = pipe(text)[0]

        return {
            "status": "success",
            "review": text,
            "label": result['label'],
            "score": float(result['score'])
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {str(e)}")

# Root endpoint to check status
@app.get("/")
def root():
    return {"status": "running", "model": "Finetuning_Longformer_IMDb_movie_reviews_Classification"}

# 7. Start the FastAPI Server in a Background Thread
def run_api():
    # Run on port 8000
    uvicorn.run(app, host="0.0.0.0", port=8000)

print("🚀 Starting FastAPI server on background thread...")
thread = threading.Thread(target=run_api, daemon=True)
thread.start()

# Give the server a couple of seconds to boot up
time.sleep(3)

# 8. Start localtunnel to expose the local server publicly for Postman testing
print("🌐 Exposing the local server to the web via localtunnel...")
# Start localtunnel in the background and pipe output to a file to extract public URL
with open("localtunnel.log", "w") as f:
    lt_process = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Wait and print the public link generated by localtunnel
time.sleep(5)
print("\n" + "="*60)
print("🔗 USE THE URL BELOW IN POSTMAN TO TEST YOUR API")
print("Add '/predict' to the end of this URL to make POST requests.")
print("="*60)

# Read first few lines of output containing the link
for i in range(5):
    line = lt_process.stdout.readline()
    if line:
        print(line.strip())

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧🤖 Loading model pipeline, please wait...


Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

✅ Model successfully loaded!
🚀 Starting FastAPI server on background thread...


INFO:     Started server process [1715]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🌐 Exposing the local server to the web via localtunnel...

🔗 USE THE URL BELOW IN POSTMAN TO TEST YOUR API
Add '/predict' to the end of this URL to make POST requests.
your url is: https://hip-buses-smell.loca.lt
